In [5]:
from pathlib import Path

ROOT = Path(
    "/Users/beck/Developer/Allstate-1C-predicting-auto-claims-severity"
)

SOURCE = ROOT / "data" / "allstate_claims_data.csv"
GATE1 = ROOT / "artifacts" / "october" / "gate 1"
MANIFEST = GATE1 / "october_partition_manifest.csv"
METADATA = GATE1 / "october_split_metadata.json"

for path in [SOURCE, MANIFEST, METADATA]:
    print(path.name, "exists:", path.is_file())

allstate_claims_data.csv exists: True
october_partition_manifest.csv exists: True
october_split_metadata.json exists: True


In [6]:
import json
import pandas as pd

manifest = pd.read_csv(MANIFEST)

with METADATA.open() as file:
    metadata = json.load(file)

print("Manifest columns:", manifest.columns.tolist())
print("\nActual partition counts:")
print(manifest["partition"].value_counts())

print("\nExpected partition counts:")
print(metadata["partition_counts"])

print("\nDuplicate IDs:", manifest["id"].duplicated().sum())
print("Missing values:", manifest.isna().sum().to_dict())

Manifest columns: ['id', 'partition']

Actual partition counts:
partition
train         141238
validation     28248
test           18832
Name: count, dtype: int64

Expected partition counts:
{'test': 18832, 'train': 141238, 'validation': 28248}

Duplicate IDs: 0
Missing values: {'id': 0, 'partition': 0}


In [7]:
import hashlib

def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


source_matches = (
    file_sha256(SOURCE) == metadata["source"]["sha256"]
)

manifest_matches = (
    file_sha256(MANIFEST)
    == metadata["artifacts_sha256"]["october_partition_manifest.csv"]
)

print("Source matches recorded version:", source_matches)
print("Manifest matches recorded version:", manifest_matches)

assert source_matches, "Source file differs from the recorded version."
assert manifest_matches, "Manifest differs from the recorded version."

Source matches recorded version: True
Manifest matches recorded version: True


In [8]:
categorical_cols = metadata["field_roles"]["categorical_predictors"]
continuous_cols = metadata["field_roles"]["continuous_predictors"]

feature_cols = categorical_cols + continuous_cols

forbidden_cols = {"id", "loss", "log1p_loss", "partition"}

assert categorical_cols == [f"cat{i}" for i in range(1, 117)]
assert continuous_cols == [f"cont{i}" for i in range(1, 15)]
assert len(set(feature_cols)) == 130
assert forbidden_cols.isdisjoint(feature_cols)

print("Categorical predictors:", len(categorical_cols))
print("Continuous predictors:", len(continuous_cols))
print("Total predictors:", len(feature_cols))
print("Forbidden columns included:", forbidden_cols.intersection(feature_cols))

Categorical predictors: 116
Continuous predictors: 14
Total predictors: 130
Forbidden columns included: set()


**STEP 2**

In [9]:
development_manifest = manifest.loc[
    manifest["partition"].isin(["train", "validation"])
]

development_ids = set(development_manifest["id"])

column_types = {
    "id": "int64",
    **{col: "string" for col in categorical_cols},
    **{col: "float64" for col in continuous_cols},
}

development_chunks = []

for chunk in pd.read_csv(
    SOURCE,
    usecols=["id"] + feature_cols,
    dtype=column_types,
    chunksize=10_000,
):
    selected = chunk.loc[chunk["id"].isin(development_ids)].copy()
    development_chunks.append(selected)

development = pd.concat(
    development_chunks, ignore_index=True
).set_index("id")

del development_chunks, chunk, selected

print("Development predictor shape:", development.shape)
print("Unique claim IDs:", development.index.is_unique)
print("Missing predictor values:", development.isna().sum().sum())
print("Columns match allowed inputs:", set(development.columns) == set(feature_cols))

Development predictor shape: (169486, 130)
Unique claim IDs: True
Missing predictor values: 0
Columns match allowed inputs: True


In [10]:
train_ids = manifest.loc[manifest["partition"].eq("train"), "id"]

valid_ids = manifest.loc[manifest["partition"].eq("validation"), "id"]

# Selects the specified claims in the manifest's order, using only the allowed columns. 
# This ensures that the training and validation sets are consistent with the manifest and that no forbidden columns are included.
X_train = development.loc[train_ids, feature_cols].copy()
X_valid = development.loc[valid_ids, feature_cols].copy()

# The assertions check that the sizes are correct, IDs remain aligned, the partitions don’t overlap, and forbidden columns are excluded.
assert X_train.shape == (141238, 130)
assert X_valid.shape == (28248, 130)

assert X_train.index.tolist() == train_ids.tolist()
assert X_valid.index.tolist() == valid_ids.tolist()

assert X_train.index.intersection(X_valid.index).empty

assert forbidden_cols.isdisjoint(X_train.columns)
assert forbidden_cols.isdisjoint(X_valid.columns)

# Only prints if every assertion passes to ensure that the training and validation sets are correctly prepared according to the manifest and metadata specifications.
print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("Claim order matches manifest: True")
print("No shared claims between training and validation: True")
print("Forbidden columns excluded: True")

Training shape: (141238, 130)
Validation shape: (28248, 130)
Claim order matches manifest: True
No shared claims between training and validation: True
Forbidden columns excluded: True


In [11]:
RARE_THRESHOLD = 30

# value_counts() counts each label within a field. 
# We then retain the labels whose counts are below 30. 
# Each field gets its own list: label "A" in cat1 is separate from label "A" in cat116.
training_category_counts = {
    col: X_train[col].value_counts()
    for col in categorical_cols
}

rare_levels = {
    col: counts.index[counts < RARE_THRESHOLD].tolist()
    for col, counts in training_category_counts.items()
}

rare_summary = pd.DataFrame({
    "training_levels": {
        col: len(counts)
        for col, counts in training_category_counts.items()
    },
    "rare_levels": {
        col: len(levels)
        for col, levels in rare_levels.items()
    },
})

print("Total rare levels:", rare_summary["rare_levels"].sum())
print("Fields with rare levels:", rare_summary["rare_levels"].gt(0).sum())

rare_summary.loc[rare_summary["rare_levels"].gt(0)].head(10)

Total rare levels: 352
Fields with rare levels: 30


,training_levels,rare_levels
cat15,2,1
cat22,2,1
cat70,2,1
cat73,3,1
cat74,3,1
cat88,4,1
cat89,8,5
cat90,7,3
cat91,8,1
cat92,7,3


In [12]:
example_col = "cat89"

example_counts = (
    training_category_counts[example_col]
    .rename_axis("original_label")
    .reset_index(name="training_count")
)

example_counts["is_rare"] = (
    example_counts["training_count"] < RARE_THRESHOLD
)

example_counts["grouped_label"] = example_counts["original_label"].where(
    ~example_counts["is_rare"],
    "__rare__",
)

example_counts

,original_label,training_count,is_rare,grouped_label
0,A,137816,False,A
1,B,3225,False,B
2,C,165,False,C
3,D,25,True,__rare__
4,E,3,True,__rare__
5,I,2,True,__rare__
6,H,1,True,__rare__
7,G,1,True,__rare__


In [13]:
rare_flags = pd.DataFrame({
    col: X_valid[col].isin(rare_levels[col])
    for col in categorical_cols
})

unseen_flags = pd.DataFrame({
    col: ~X_valid[col].isin(training_category_counts[col].index)
    for col in categorical_cols
})

print(
    "Validation claims with any training-rare label:",
    rare_flags.any(axis=1).sum(),
)

print(
    "Validation claims with any unseen label:",
    unseen_flags.any(axis=1).sum(),
)

print("Total unseen field occurrences:", unseen_flags.sum().sum())

unseen_by_field = unseen_flags.sum()
unseen_by_field.loc[unseen_by_field.gt(0)]

Validation claims with any training-rare label: 490
Validation claims with any unseen label: 12
Total unseen field occurrences: 13


cat105    1
cat107    1
cat109    1
cat110    1
cat114    1
cat116    8
dtype: int64

In [14]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])

for package in ["pandas", "numpy", "scikit-learn", "xgboost"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

Python: 3.13.2
pandas: 2.3.2
numpy: 2.3.3
scikit-learn: 1.7.2
xgboost: not installed


In [15]:
%pip install scikit-learn==1.7.2


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin

print("scikit-learn:", sklearn.__version__)
print("Transformer imports successful")

scikit-learn: 1.7.2
Transformer imports successful


In [35]:
from sklearn.utils.validation import check_is_fitted


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=30):
        self.min_count = min_count

    def fit(self, X, y=None):

        if X.eq("__rare__").any().any():
            raise ValueError("The reserved label '__rare__' already exists.")

        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        self.known_levels_ = {}
        self.rare_levels_ = {}

        for col in self.feature_names_in_:
            counts = X[col].value_counts(dropna=True)

            self.known_levels_[col] = counts.index.tolist()
            self.rare_levels_[col] = counts.index[
                counts < self.min_count
            ].tolist()

        return self

    def transform(self, X):
        check_is_fitted(self, ["known_levels_", "rare_levels_"])

        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError(
                "Categorical columns or their order differ from fitting."
            )

        transformed = X.astype("string").copy()

        for col in self.feature_names_in_:
            rare = self.rare_levels_[col]

            if rare:
                is_rare = transformed[col].isin(rare)
                is_unseen = ~transformed[col].isin(
                    self.known_levels_[col]
                )

                transformed[col] = transformed[col].mask(
                    is_rare | is_unseen,
                    "__rare__",
                )

            transformed[col] = transformed[col].fillna("__unseen__")

        return transformed

In [21]:
category_grouper = RareCategoryGrouper(min_count=RARE_THRESHOLD)

train_categories = category_grouper.fit_transform(
    X_train[categorical_cols]
)

valid_categories = category_grouper.transform(
    X_valid[categorical_cols]
)

print("Training categories shape:", train_categories.shape)
print("Validation categories shape:", valid_categories.shape)

print("\nGrouped training counts for cat89:")
print(train_categories["cat89"].value_counts())

assert train_categories.index.equals(X_train.index)
assert valid_categories.index.equals(X_valid.index)

Training categories shape: (141238, 116)
Validation categories shape: (28248, 116)

Grouped training counts for cat89:
cat89
A           137816
B             3225
C              165
__rare__        32
Name: count, dtype: Int64


In [22]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    drop=None,
    sparse_output=True,
    dtype="float64",
)

train_encoded = onehot_encoder.fit_transform(train_categories)
valid_encoded = onehot_encoder.transform(valid_categories)

print("Encoded training shape:", train_encoded.shape)
print("Encoded validation shape:", valid_encoded.shape)
print("Sparse format:", train_encoded.format)

assert train_encoded.shape[0] == len(X_train)
assert valid_encoded.shape[0] == len(X_valid)
assert train_encoded.shape[1] == valid_encoded.shape[1]

Encoded training shape: (141238, 794)
Encoded validation shape: (28248, 794)
Sparse format: csr


**Step 3 - Combining grouping and encoding into one pipeline**

In [24]:
from sklearn.pipeline import Pipeline

categorical_pipeline = Pipeline([
    (
        "rare_grouping",
        RareCategoryGrouper(min_count=RARE_THRESHOLD),
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            drop=None,
            sparse_output=True,
            dtype="float64",
        ),
    ),
])

# During fit_transform(), the pipeline:
# 1. Learns rare category rules from training and applies them.
# 2. Learns one-hot columns from that grouped training data and encodes it.
train_cat_pipeline = categorical_pipeline.fit_transform(
    X_train[categorical_cols]
)

# During validation transform(), it applies both sets of learned rules unchanged.
valid_cat_pipeline = categorical_pipeline.transform(
    X_valid[categorical_cols]
)

print("Pipeline training shape:", train_cat_pipeline.shape)
print("Pipeline validation shape:", valid_cat_pipeline.shape)

# The final assertions compare the pipeline’s results with our earlier results. 
# .nnz counts nonzero entries
# Zero differences means the outputs match exactly.
assert (train_cat_pipeline != train_encoded).nnz == 0
assert (valid_cat_pipeline != valid_encoded).nnz == 0

print("Pipeline matches the separate steps: True")

Pipeline training shape: (141238, 794)
Pipeline validation shape: (28248, 794)
Pipeline matches the separate steps: True


In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

glm_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, categorical_cols),
        ("continuous", StandardScaler(), continuous_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

X_train_processed = glm_preprocessor.fit_transform(X_train)
X_valid_processed = glm_preprocessor.transform(X_valid)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_valid_processed.shape)
print("Output format:", X_train_processed.format)

assert X_train_processed.shape == (141238, 808)
assert X_valid_processed.shape == (28248, 808)

print("Combined feature counts verified: True")

Processed training shape: (141238, 808)
Processed validation shape: (28248, 808)
Output format: csr
Combined feature counts verified: True


In [ ]:
import numpy as np

fitted_scaler = glm_preprocessor.named_transformers_["continuous"]

expected_means = X_train[continuous_cols].mean().to_numpy()

# ddof=0 calculates the population standard deviation, which is the convention StandardScaler uses. 
# Pandas defaults to ddof=1.
expected_scales = X_train[continuous_cols].std(ddof=0).to_numpy()

assert np.allclose(fitted_scaler.mean_, expected_means)
assert np.allclose(fitted_scaler.scale_, expected_scales)
assert fitted_scaler.n_samples_seen_ == len(X_train)

# .data contains the explicitly stored values in our sparse matrices. The other entries are zeros, which are already finite.
assert np.isfinite(X_train_processed.data).all()
assert np.isfinite(X_valid_processed.data).all()

print("Scaler means match training means: True")
print("Scaler scales match training standard deviations: True")
print("Rows used to fit scaler:", fitted_scaler.n_samples_seen_)
print("All processed training values are finite: True")
print("All processed validation values are finite: True")

Scaler means match training means: True
Scaler scales match training standard deviations: True
Rows used to fit scaler: 141238
All processed training values are finite: True
All processed validation values are finite: True


In [27]:
unseen_examples = X_train.iloc[:len(categorical_cols)].copy()

for row_number, col in enumerate(categorical_cols):
    made_up_label = f"__test_unseen_{col}__"

    assert made_up_label not in training_category_counts[col].index

    column_number = unseen_examples.columns.get_loc(col)
    unseen_examples.iloc[row_number, column_number] = made_up_label

unseen_processed = glm_preprocessor.transform(unseen_examples)

assert unseen_processed.shape == (116, 808)
assert np.isfinite(unseen_processed.data).all()

print("Categorical fields tested:", len(categorical_cols))
print("Processed shape:", unseen_processed.shape)
print("All test rows retained: True")
print("All processed values are finite: True")

Categorical fields tested: 116
Processed shape: (116, 808)
All test rows retained: True
All processed values are finite: True


In [28]:
fitted_categories = glm_preprocessor.named_transformers_["categorical"]
fitted_grouper = fitted_categories.named_steps["rare_grouping"]
fitted_encoder = fitted_categories.named_steps["onehot"]

start = 0
rare_fallback_checks = 0
zero_fallback_checks = 0

for row_number, col in enumerate(categorical_cols):
    levels = fitted_encoder.categories_[row_number]
    stop = start + len(levels)

    actual = unseen_processed[
        row_number, start:stop
    ].toarray().ravel()

    expected = np.zeros(len(levels))

    if fitted_grouper.rare_levels_[col]:
        rare_position = list(levels).index("__rare__")
        expected[rare_position] = 1
        rare_fallback_checks += 1
    else:
        zero_fallback_checks += 1

    np.testing.assert_array_equal(
        actual,
        expected,
        err_msg=f"Unexpected unseen-label encoding for {col}",
    )

    start = stop

assert start == 794
assert rare_fallback_checks == 30
assert zero_fallback_checks == 86

print("Unseen labels mapped to rare group:", rare_fallback_checks)
print("Unseen labels encoded as all zeros:", zero_fallback_checks)
print("All fallback encodings verified: True")

Unseen labels mapped to rare group: 30
Unseen labels encoded as all zeros: 86
All fallback encodings verified: True


In [29]:
def validate_feature_schema(X, expected_columns, continuous_columns):
    if not isinstance(X, pd.DataFrame):
        raise TypeError("Inputs must be a pandas DataFrame.")

    if not X.columns.is_unique:
        raise ValueError("Duplicate column names are not allowed.")

    missing = sorted(set(expected_columns) - set(X.columns))
    unexpected = sorted(set(X.columns) - set(expected_columns))

    if missing or unexpected:
        raise ValueError(
            f"Schema mismatch. Missing: {missing}; "
            f"unexpected: {unexpected}"
        )

    for col in continuous_columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            raise ValueError(f"{col} must contain numeric values.")

    continuous_values = X[continuous_columns].to_numpy(
        dtype="float64",
        na_value=np.nan,
    )

    if not np.isfinite(continuous_values).all():
        raise ValueError(
            "Continuous predictors contain missing or infinite values."
        )

In [30]:
validate_feature_schema(X_train, feature_cols, continuous_cols)
validate_feature_schema(X_valid, feature_cols, continuous_cols)

print("Training inputs pass schema checks: True")
print("Validation inputs pass schema checks: True")

Training inputs pass schema checks: True
Validation inputs pass schema checks: True


In [31]:
class FeatureSchemaValidator(BaseEstimator, TransformerMixin):
    def __init__(self, expected_columns, continuous_columns):
        self.expected_columns = expected_columns
        self.continuous_columns = continuous_columns

    def fit(self, X, y=None):
        validate_feature_schema(
            X, self.expected_columns, self.continuous_columns
        )

        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        return self

    def transform(self, X):
        check_is_fitted(self, ["feature_names_in_"])

        validate_feature_schema(
            X, self.expected_columns, self.continuous_columns
        )

        return X.loc[:, self.expected_columns].copy()

In [32]:
glm_preprocessing_pipeline = Pipeline([
    (
        "schema",
        FeatureSchemaValidator(
            expected_columns=feature_cols,
            continuous_columns=continuous_cols,
        ),
    ),
    ("preprocessing", glm_preprocessor),
])

X_train_checked = glm_preprocessing_pipeline.fit_transform(X_train)
X_valid_checked = glm_preprocessing_pipeline.transform(X_valid)

assert X_train_checked.shape == X_train_processed.shape
assert X_valid_checked.shape == X_valid_processed.shape

assert (X_train_checked != X_train_processed).nnz == 0
assert (X_valid_checked != X_valid_processed).nnz == 0

print("Training shape:", X_train_checked.shape)
print("Validation shape:", X_valid_checked.shape)
print("Schema checks integrated: True")
print("Outputs match previous preprocessing: True")

Training shape: (141238, 808)
Validation shape: (28248, 808)
Schema checks integrated: True
Outputs match previous preprocessing: True


In [33]:
sample = X_valid.head(3).copy()

invalid_examples = {
    "missing column": sample.drop(columns=["cont1"]),
    "renamed column": sample.rename(columns={"cat1": "renamed_cat1"}),
    "duplicate column": pd.concat([sample, sample[["cont1"]]], axis=1),
}

for col in ["id", "loss", "log1p_loss", "partition"]:
    invalid_examples[f"forbidden column: {col}"] = sample.assign(
        **{col: 0}
    )

for label, value in [
    ("missing continuous value", np.nan),
    ("infinite continuous value", np.inf),
]:
    example = sample.copy()
    example.loc[example.index[0], "cont1"] = value
    invalid_examples[label] = example

for label, example in invalid_examples.items():
    try:
        glm_preprocessing_pipeline.transform(example)
    except ValueError as error:
        print(f"PASS — {label}: {error}")
    else:
        raise AssertionError(f"Invalid input was accepted: {label}")

PASS — missing column: Schema mismatch. Missing: ['cont1']; unexpected: []
PASS — renamed column: Schema mismatch. Missing: ['cat1']; unexpected: ['renamed_cat1']
PASS — duplicate column: Duplicate column names are not allowed.
PASS — forbidden column: id: Schema mismatch. Missing: []; unexpected: ['id']
PASS — forbidden column: loss: Schema mismatch. Missing: []; unexpected: ['loss']
PASS — forbidden column: log1p_loss: Schema mismatch. Missing: []; unexpected: ['log1p_loss']
PASS — forbidden column: partition: Schema mismatch. Missing: []; unexpected: ['partition']
PASS — missing continuous value: Continuous predictors contain missing or infinite values.
PASS — infinite continuous value: Continuous predictors contain missing or infinite values.


In [34]:
missing_examples = unseen_examples.copy()

for row_number, col in enumerate(categorical_cols):
    column_number = missing_examples.columns.get_loc(col)
    missing_examples.iloc[row_number, column_number] = pd.NA

missing_processed = glm_preprocessing_pipeline.transform(
    missing_examples
)

unseen_reference = glm_preprocessing_pipeline.transform(
    unseen_examples
)

assert missing_processed.shape == (116, 808)
assert np.isfinite(missing_processed.data).all()
assert (missing_processed != unseen_reference).nnz == 0

print("Categorical fields tested:", len(categorical_cols))
print("All test rows retained: True")
print("All processed values are finite: True")
print("Missing labels behave like unseen labels: True")

Categorical fields tested: 116
All test rows retained: True
All processed values are finite: True
Missing labels behave like unseen labels: True


In [36]:
training_with_missing = (
    X_train[categorical_cols].head(100).copy()
)

training_with_missing.loc[
    training_with_missing.index[0], "cat89"
] = pd.NA

probe_grouper = RareCategoryGrouper(min_count=RARE_THRESHOLD)
probe_grouped = probe_grouper.fit_transform(training_with_missing)

assert all(
    pd.notna(label)
    for label in probe_grouper.known_levels_["cat89"]
)

assert probe_grouped.shape == training_with_missing.shape
assert not probe_grouped.isna().any().any()

print("Fitting accepts missing categorical labels: True")
print("Missing values excluded from learned vocabulary: True")
print("Grouped output contains no missing values: True")

Fitting accepts missing categorical labels: True
Missing values excluded from learned vocabulary: True
Grouped output contains no missing values: True


In [37]:
example_training = pd.DataFrame({
    "example_cat": ["A", "A", "B", pd.NA]
})

example_grouper = RareCategoryGrouper(min_count=1)
example_grouped = example_grouper.fit_transform(example_training)

example_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

example_encoded = example_encoder.fit_transform(example_grouped)

print("Grouped labels:")
print(example_grouped)

print("\nEncoder's learned categories:")
print(example_encoder.categories_[0])

print("\nEncoding of the missing-label row:")
print(example_encoded[-1])

Grouped labels:
  example_cat
0           A
1           A
2           B
3  __unseen__

Encoder's learned categories:
['A' 'B' '__unseen__']

Encoding of the missing-label row:
[0. 0. 1.]


In [38]:
from scipy.sparse import csr_matrix


class PlaceholderSafeOneHotEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        self.active_columns_ = []
        categories = []

        for col in self.feature_names_in_:
            levels = sorted(
                label
                for label in X[col].unique()
                if label != "__unseen__"
            )

            if levels:
                self.active_columns_.append(col)
                categories.append(levels)

        self.encoder_ = None

        if self.active_columns_:
            self.encoder_ = OneHotEncoder(
                categories=categories,
                handle_unknown="ignore",
                drop=None,
                sparse_output=True,
                dtype="float64",
            )

            self.encoder_.fit(X[self.active_columns_])

        return self

    def transform(self, X):
        check_is_fitted(self, ["active_columns_", "encoder_"])

        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError(
                "Categorical columns or their order differ from fitting."
            )

        if self.encoder_ is None:
            return csr_matrix((len(X), 0), dtype="float64")

        return self.encoder_.transform(X[self.active_columns_])

In [39]:
safe_example_encoder = PlaceholderSafeOneHotEncoder()
safe_example_encoded = safe_example_encoder.fit_transform(
    example_grouped
)

print("Learned categories:")
print(safe_example_encoder.encoder_.categories_[0])

print("\nEncoding of the missing-label row:")
print(safe_example_encoded[-1].toarray().ravel())

assert safe_example_encoded.shape == (4, 2)
assert safe_example_encoded[-1].nnz == 0

print("\nMissing-label fallback verified: True")

Learned categories:
['A' 'B']

Encoding of the missing-label row:
[0. 0.]

Missing-label fallback verified: True


In [40]:
categorical_pipeline = Pipeline([
    (
        "rare_grouping",
        RareCategoryGrouper(min_count=RARE_THRESHOLD),
    ),
    ("onehot", PlaceholderSafeOneHotEncoder()),
])

glm_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, categorical_cols),
        ("continuous", StandardScaler(), continuous_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

glm_preprocessing_pipeline = Pipeline([
    (
        "schema",
        FeatureSchemaValidator(
            expected_columns=feature_cols,
            continuous_columns=continuous_cols,
        ),
    ),
    ("preprocessing", glm_preprocessor),
])

X_train_checked = glm_preprocessing_pipeline.fit_transform(X_train)
X_valid_checked = glm_preprocessing_pipeline.transform(X_valid)

assert X_train_checked.shape == (141238, 808)
assert X_valid_checked.shape == (28248, 808)

assert np.isfinite(X_train_checked.data).all()
assert np.isfinite(X_valid_checked.data).all()

assert (X_train_checked != X_train_processed).nnz == 0
assert (X_valid_checked != X_valid_processed).nnz == 0

print("Training shape:", X_train_checked.shape)
print("Validation shape:", X_valid_checked.shape)
print("All processed values are finite: True")
print("Outputs match the original results: True")

Training shape: (141238, 808)
Validation shape: (28248, 808)
All processed values are finite: True
Outputs match the original results: True


In [41]:
unseen_checked = glm_preprocessing_pipeline.transform(
    unseen_examples
)

missing_checked = glm_preprocessing_pipeline.transform(
    missing_examples
)

assert unseen_checked.shape == (116, 808)
assert missing_checked.shape == (116, 808)

assert np.isfinite(unseen_checked.data).all()
assert np.isfinite(missing_checked.data).all()

assert (unseen_checked != unseen_processed).nnz == 0
assert (missing_checked != unseen_checked).nnz == 0

print("Unseen-label encodings match verified fallbacks: True")
print("Missing labels behave like unseen labels: True")
print("All example rows retained with finite values: True")

rejected_count = 0

for label, example in invalid_examples.items():
    try:
        glm_preprocessing_pipeline.transform(example)
    except ValueError:
        rejected_count += 1
    else:
        raise AssertionError(f"Invalid input was accepted: {label}")

assert rejected_count == 9

print("Invalid inputs correctly rejected:", rejected_count)

Unseen-label encodings match verified fallbacks: True
Missing labels behave like unseen labels: True
All example rows retained with finite values: True
Invalid inputs correctly rejected: 9


In [42]:
from sklearn.base import clone

fold_example = X_train.head(60).copy()
fold_example["cat89"] = ["A"] * 30 + ["B"] * 30

first_pipeline = clone(glm_preprocessing_pipeline)
second_pipeline = clone(glm_preprocessing_pipeline)

# First subset: 29 A labels and 30 B labels.
first_pipeline.fit(fold_example.iloc[1:])

# Second subset: 30 A labels and 29 B labels.
second_pipeline.fit(fold_example.iloc[:-1])

first_grouper = (
    first_pipeline.named_steps["preprocessing"]
    .named_transformers_["categorical"]
    .named_steps["rare_grouping"]
)

second_grouper = (
    second_pipeline.named_steps["preprocessing"]
    .named_transformers_["categorical"]
    .named_steps["rare_grouping"]
)

assert first_grouper.rare_levels_["cat89"] == ["A"]
assert second_grouper.rare_levels_["cat89"] == ["B"]

print("First subset's rare labels:", first_grouper.rare_levels_["cat89"])
print("Second subset's rare labels:", second_grouper.rare_levels_["cat89"])
print("Independent training-subset learning verified: True")

First subset's rare labels: ['A']
Second subset's rare labels: ['B']
Independent training-subset learning verified: True


In [43]:
import pickle

state_before = hashlib.sha256(
    pickle.dumps(glm_preprocessing_pipeline)
).hexdigest()

validation_check = glm_preprocessing_pipeline.transform(X_valid)

state_after = hashlib.sha256(
    pickle.dumps(glm_preprocessing_pipeline)
).hexdigest()

assert state_before == state_after
assert validation_check.shape == (28248, 808)
assert np.isfinite(validation_check.data).all()
assert (validation_check != X_valid_checked).nnz == 0

print("Fitted pipeline unchanged after validation: True")
print("Validation output matches previous results: True")
print("All validation rows retained with finite values: True")

Fitted pipeline unchanged after validation: True
Validation output matches previous results: True
All validation rows retained with finite values: True


In [44]:
%pip install xgboost==3.1.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 5.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
